# GPU vs CPU Benchmark for Whisper

This notebook benchmarks OpenAI Whisper transcription performance on GPU vs CPU to verify GPU acceleration is working correctly.

In [ ]:
# Cell 1: Environment Check
# Check if CUDA is available and display GPU information

import torch
import whisper
import time

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU detected - benchmark will only test CPU performance")

In [ ]:
# Cell 2: Load Whisper Model
# Load the "small" model (can change to tiny, base, medium, large as needed)

AUDIO_FILE = "../data/US_DebateAudio.wav" 
MODEL_SIZE = "small"

print(f"Loading Whisper '{MODEL_SIZE}' model...")
model = whisper.load_model(MODEL_SIZE)
print("Model loaded successfully")

In [ ]:
# Cell 3: CPU Transcription Benchmark
# Run transcription on CPU and measure time

print("Running CPU benchmark...")
model = model.cpu()
torch.cuda.empty_cache()

start_time = time.time()
cpu_result = whisper.transcribe(model, AUDIO_FILE)
cpu_time = time.time() - start_time

print(f"CPU Transcription completed in {cpu_time:.2f} seconds")
print(f"Detected language: {cpu_result['language']}")
print(f"Text preview: {cpu_result['text'][:100]}...")

In [ ]:
# Cell 4: GPU Transcription Benchmark
# Run transcription on GPU and measure time (if GPU available)

if torch.cuda.is_available():
    print("Running GPU benchmark...")
    model = model.to("cuda")
    torch.cuda.empty_cache()
    
    start_time = time.time()
    gpu_result = whisper.transcribe(model, AUDIO_FILE)
    gpu_time = time.time() - start_time
    
    print(f"GPU Transcription completed in {gpu_time:.2f} seconds")
    print(f"Detected language: {gpu_result['language']}")
    print(f"Text preview: {gpu_result['text'][:100]}...")
else:
    print("GPU not available - skipping GPU benchmark")
    gpu_time = None

In [ ]:
# Cell 5: Side-by-Side Comparison
# Display performance comparison and speedup

print("\n" + "="*50)
print("BENCHMARK RESULTS")
print("="*50)
print(f"CPU Time: {cpu_time:.2f} seconds")

if gpu_time is not None:
    print(f"GPU Time: {gpu_time:.2f} seconds")
    speedup = cpu_time / gpu_time
    print(f"Speedup: {speedup:.2f}x faster on GPU")
    print("\n" + "="*50)
    print("PERFORMANCE INTERPRETATION")
    print("="*50)
    if speedup > 3:
        print("Excellent GPU acceleration!")
    elif speedup > 1.5:
        print("Good GPU acceleration - GPU is providing meaningful speedup")
    elif speedup > 1:
        print("Moderate GPU acceleration - consider checking CUDA configuration")
    else:
        print("Warning: GPU slower than CPU - check CUDA drivers and configuration")
else:
    print("GPU benchmark not available")
    print("\nTo enable GPU acceleration:")
    print("1. Install CUDA toolkit")
    print("2. Install PyTorch with CUDA support")
    print("3. Restart this notebook")

# Speaker Diarization Testing

Testing speaker diarization using pyannote.audio and WhisperX on GPU.

In [ ]:
# Cell 6: Install Required Packages for Diarization
# Install pyannote.audio and WhisperX if not already installed

!pip install pyannote.audio
!pip install whisperx

In [ ]:
# Cell 7: Setup Pyannote Audio Pipeline
# Initialize pyannote.audio for speaker diarization
# You'll need a Hugging Face token for this

from pyannote.audio import Pipeline
import torch

from dotenv import load_dotenv
import os
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Load the diarization pipeline
print("Loading pyannote.audio diarization pipeline...")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN
)

# Move to GPU if available
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))
    print("Diarization pipeline loaded on GPU")
else:
    print("Diarization pipeline loaded on CPU")

print("Pipeline ready!")

In [ ]:
# Cell 8: Run Pyannote Diarization
# Perform speaker diarization on the audio file

print(f"Running diarization on {AUDIO_FILE}...")
start_time = time.time()

diarization = diarization_pipeline(AUDIO_FILE)

diarization_time = time.time() - start_time
print(f"Diarization completed in {diarization_time:.2f} seconds")

# Display the results
print("\nSpeaker Timeline:")
print("="*60)
for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"Speaker {speaker}: {turn.start:.1f}s - {turn.end:.1f}s")
    
print("="*60)
print(f"Total unique speakers detected: {len(set([s for _, _, s in diarization.itertracks(yield_label=True)]))})")

## WhisperX - Combined Transcription & Diarization

In [ ]:
# Cell 9: Load WhisperX Model
# WhisperX provides faster transcription with word-level timestamps

import whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading WhisperX model on {device}...")
whisperx_model = whisperx.load_model("small", device, compute_type=compute_type)
print("WhisperX model loaded successfully")

In [ ]:
# Cell 10: Transcribe with WhisperX
# Get initial transcription with word-level timestamps

print("Transcribing with WhisperX...")
start_time = time.time()

audio = whisperx.load_audio(AUDIO_FILE)
result = whisperx_model.transcribe(audio, batch_size=16)

transcribe_time = time.time() - start_time
print(f"Transcription completed in {transcribe_time:.2f} seconds")
print(f"Detected language: {result['language']}")
print(f"\nTranscript preview: {result['segments'][0]['text'][:200]}..." if result['segments'] else "No segments found")

In [ ]:
# Cell 11: Align Whisper Output
# Align whisper output to get accurate word-level timestamps

print("Aligning transcript...")
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

print("Alignment completed")
print(f"Total segments: {len(result['segments'])}")

In [ ]:
# Cell 12: Perform Speaker Diarization with WhisperX
# Assign speaker labels to each segment

print("Performing speaker diarization...")
start_time = time.time()

diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
diarize_segments = diarize_model(audio)

# Assign speakers to segments
result = whisperx.assign_word_speakers(diarize_segments, result)

diarize_time = time.time() - start_time
print(f"Diarization completed in {diarize_time:.2f} seconds")

# Display results with speaker labels
print("\nTranscript with Speakers:")
print("="*60)
for segment in result["segments"][:10]:  # Show first 10 segments
    speaker = segment.get('speaker', 'Unknown')
    text = segment['text']
    start = segment['start']
    end = segment['end']
    print(f"[{start:.1f}s - {end:.1f}s] {speaker}: {text}")
print("="*60)

In [ ]:
# Cell 13: Summary of Diarization Results
# Compare pyannote vs WhisperX performance

print("\n" + "="*60)
print("DIARIZATION BENCHMARK SUMMARY")
print("="*60)
print(f"\nPyannote.audio:")
print(f"  - Diarization time: {diarization_time:.2f} seconds")
print(f"  - Speakers detected: {len(set([s for _, _, s in diarization.itertracks(yield_label=True)]))}")
print(f"  - GPU accelerated: {'Yes' if torch.cuda.is_available() else 'No'}")

print(f"\nWhisperX:")
print(f"  - Transcription time: {transcribe_time:.2f} seconds")
print(f"  - Diarization time: {diarize_time:.2f} seconds")
print(f"  - Total time: {transcribe_time + diarize_time:.2f} seconds")
print(f"  - Segments with speakers: {len([s for s in result['segments'] if 'speaker' in s])}")
print(f"  - GPU accelerated: {'Yes' if device == 'cuda' else 'No'}")

print("\n" + "="*60)
print("RECOMMENDATION:")
print("="*60)
if torch.cuda.is_available():
    print("✓ Both methods support GPU acceleration")
    print("✓ Use pyannote for diarization-only tasks")
    print("✓ Use WhisperX for combined transcription + diarization")
else:
    print("⚠ GPU not available - consider enabling GPU for faster processing")
print("="*60)